# MediaPipe posture timeline notebook

Este notebook implementa um protótipo de triagem geométrica de postura para vídeos com enquadramento de tronco e cabeça. Ele foi dividido em etapas pequenas para facilitar leitura, testes e ajustes de limiares.

Pontos importantes antes de começar:
- O pipeline reporta apenas sinais observáveis de postura e self-touch.
- As saídas finais do JSON e do vídeo anotado ficam em **English**, como definido no plano.
- O notebook **não** infere emoção, culpa, agressividade, diagnóstico ou estado mental.
- A implementação usa **MediaPipe Tasks** com **PoseLandmarker + HandLandmarker** e amostragem aproximada de **5 FPS**.


## 1) Instalação opcional do ambiente

Execute a próxima célula apenas se o ambiente ainda não tiver as dependências necessárias. Se você já estiver com `opencv-python`, `mediapipe`, `numpy`, `pandas` e `tqdm` instalados, pode pular esta etapa. Os modelos `.task` são resolvidos pelo notebook e o modelo de mãos é baixado automaticamente se estiver faltando.


In [1]:
# Rode esta célula uma vez por ambiente, se necessário.
%pip install -q opencv-python mediapipe numpy pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Imports e checagem do runtime

Esta etapa importa as bibliotecas e mostra um resumo rápido do ambiente. Se faltar alguma dependência, instale o projeto com `pip install -e .` para garantir o processamento completo.


In [2]:
from __future__ import annotations

import json
import math
import urllib.request
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping

import cv2
import numpy as np
from IPython.display import Markdown, display
from tqdm.auto import tqdm

try:
    import mediapipe as mp
except ImportError:
    mp = None

try:
    import pandas as pd
except ImportError:
    pd = None

display(Markdown('\n'.join([
    '### Runtime check',
    f'- OpenCV version: `{cv2.__version__}`',
    f'- NumPy version: `{np.__version__}`',
    f'- MediaPipe available: `{mp is not None}`',
    f'- Pandas available: `{pd is not None}`',
    f'- tqdm available: `{tqdm is not None}`',
])))

if mp is None:
    display(Markdown(
        '**MediaPipe is missing.** Execute the installation cell above and rerun the notebook before processing videos.'
    ))


### Runtime check
- OpenCV version: `4.13.0`
- NumPy version: `2.4.4`
- MediaPipe available: `True`
- Pandas available: `True`
- tqdm available: `True`

## 3) Configuração do notebook

Aqui ficam os caminhos principais, os limiares, os pesos e os assets de modelo usados na triagem. Como o `mediapipe` atual no Python 3.13 expõe a API `Tasks`, a configuração tamb?m explicita os arquivos `.task` usados por pose e mãos.


In [3]:
# Resolve a pasta `concepts_video` mesmo quando o notebook for aberto a partir da raiz do repositório.
def locate_concepts_video_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if candidate.name == 'concepts_video' and (candidate / 'data').exists():
            return candidate
        nested = candidate / 'concepts_video'
        if nested.exists() and (nested / 'data').exists():
            return nested.resolve()
    return start


NOTEBOOK_ROOT = locate_concepts_video_root()
REPO_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == 'concepts_video' else NOTEBOOK_ROOT
ARTIFACTS_DIR = NOTEBOOK_ROOT / 'artifacts'
INPUT_VIDEO_DIR = NOTEBOOK_ROOT / 'data' / 'video'
OUTPUT_JSON_DIR = NOTEBOOK_ROOT / 'outputs' / 'posture_timelines'
OUTPUT_VIDEO_DIR = NOTEBOOK_ROOT / 'outputs' / 'annotated_videos'

POSE_LANDMARKER_MODEL_PATH = ARTIFACTS_DIR / 'pose_landmarker_heavy.task'
HAND_LANDMARKER_MODEL_PATH = ARTIFACTS_DIR / 'hand_landmarker.task'
POSE_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task'
HAND_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'

SUPPORTED_VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.m4v'}

TARGET_SAMPLE_FPS = 5.0
VISIBILITY_THRESHOLD = 0.50
EMA_ALPHA = 0.35
EMA_MAX_GAP_FRAMES = 3
WINDOW_SECONDS = 8.0
WINDOW_STRIDE_SECONDS = 2.0
MIN_SHOULDER_WIDTH_NORM = 0.02

PIPELINE_BACKEND = 'MediaPipe Tasks (PoseLandmarker + HandLandmarker)'
SCHEMA_VERSION = 'posture_signal_timeline_v1'
COORDINATE_MODE_WORLD = 'mixed_world_pose_plus_2d_hands'
COORDINATE_MODE_2D = 'image_2d_only'

RULE_WEIGHTS = {
    'hand_on_face': 1.0,
    'hand_on_neck': 0.9,
    'hand_on_chest': 0.8,
    'head_down': 0.7,
    'forward_head': 0.6,
    'rounded_shoulders_or_asymmetry': 0.4,
}

RULE_DISPLAY_NAMES = {
    'hand_on_face': 'hand-to-face contact',
    'hand_on_neck': 'hand-to-neck contact',
    'hand_on_chest': 'hand-to-chest contact',
    'head_down': 'head down',
    'forward_head': 'forward head',
    'rounded_shoulders_or_asymmetry': 'upper-body tension / asymmetry',
}

FRAME_LEVEL_COLORS = {
    'ok': (80, 170, 80),
    'possible_signal': (0, 180, 255),
    'strong_signal': (0, 90, 255),
    'insufficient_data': (120, 120, 120),
}

WINDOW_LEVEL_SEVERITY = {
    'insufficient_data': -1,
    'ok': 0,
    'possible_signal': 1,
    'strong_signal': 2,
}

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_VIDEO_PATHS = []
if INPUT_VIDEO_DIR.exists():
    DEFAULT_VIDEO_PATHS = sorted(
        path for path in INPUT_VIDEO_DIR.iterdir() if path.suffix.lower() in SUPPORTED_VIDEO_EXTENSIONS
    )

display(Markdown('\n'.join([
    '### Current configuration',
    f'- Notebook root: `{NOTEBOOK_ROOT}`',
    f'- Input video dir: `{INPUT_VIDEO_DIR}`',
    f'- Pose model path: `{POSE_LANDMARKER_MODEL_PATH}`',
    f'- Hand model path: `{HAND_LANDMARKER_MODEL_PATH}`',
    f'- JSON output dir: `{OUTPUT_JSON_DIR}`',
    f'- Annotated MP4 dir: `{OUTPUT_VIDEO_DIR}`',
    f'- Default videos found: `{len(DEFAULT_VIDEO_PATHS)}`',
    f'- Sample target FPS: `{TARGET_SAMPLE_FPS:.1f}`',
    f'- EMA alpha: `{EMA_ALPHA:.2f}`',
    f'- Window size / stride: `{WINDOW_SECONDS:.1f}s / {WINDOW_STRIDE_SECONDS:.1f}s`',
])))


### Current configuration
- Notebook root: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video`
- Input video dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\data\video`
- Pose model path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\pose_landmarker_heavy.task`
- Hand model path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\hand_landmarker.task`
- JSON output dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\posture_timelines`
- Annotated MP4 dir: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\annotated_videos`
- Default videos found: `4`
- Sample target FPS: `5.0`
- EMA alpha: `0.35`
- Window size / stride: `8.0s / 2.0s`

## 4) Download dos modelos `.task`

Esta célula existe para baixar explicitamente os modelos oficiais usados pelo notebook. A ideia é evitar downloads implícitos durante o processamento do vídeo: primeiro você resolve os assets, depois roda a análise.


In [4]:
# Baixa um asset remoto para o disco exibindo uma barra de progresso simples.
def download_file_with_progress(url: str, destination_path: Path) -> None:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.urlopen(url)
    total_bytes = int(request.headers.get('Content-Length') or 0)
    temp_path = destination_path.with_suffix(destination_path.suffix + '.tmp')

    try:
        with request, temp_path.open('wb') as file_handle, tqdm(
            total=total_bytes or None,
            desc=f'Downloading {destination_path.name}',
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as progress_bar:
            while True:
                chunk = request.read(1024 * 1024)
                if not chunk:
                    break
                file_handle.write(chunk)
                progress_bar.update(len(chunk))
        temp_path.replace(destination_path)
    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


# Garante que os modelos `.task` necessários estejam presentes na pasta de artifacts.
def download_required_model_assets(force_download: bool = False) -> dict[str, Path]:
    model_specs = [
        ('pose', POSE_LANDMARKER_MODEL_PATH, POSE_LANDMARKER_MODEL_URL),
        ('hand', HAND_LANDMARKER_MODEL_PATH, HAND_LANDMARKER_MODEL_URL),
    ]
    resolved_paths: dict[str, Path] = {}

    for model_name, model_path, model_url in model_specs:
        if model_path.exists() and not force_download:
            resolved_paths[model_name] = model_path
            continue
        download_file_with_progress(model_url, model_path)
        resolved_paths[model_name] = model_path

    return resolved_paths


# Valida se os assets já existem antes de iniciar o processamento de vídeos.
def assert_model_assets_available() -> None:
    missing_assets = [
        model_path.name
        for model_path in [POSE_LANDMARKER_MODEL_PATH, HAND_LANDMARKER_MODEL_PATH]
        if not model_path.exists()
    ]
    if missing_assets:
        raise FileNotFoundError(
            'Missing model assets: '
            + ', '.join(missing_assets)
            + '. Run the model download cell before processing videos.'
        )


MODEL_ASSETS = download_required_model_assets(force_download=False)
display(Markdown('\n'.join([
    '### Model assets ready',
    f'- Pose task: `{MODEL_ASSETS["pose"]}`',
    f'- Hand task: `{MODEL_ASSETS["hand"]}`',
])))


### Model assets ready
- Pose task: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\pose_landmarker_heavy.task`
- Hand task: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\artifacts\hand_landmarker.task`

## 5) Modelos de dados e extração de landmarks

O MediaPipe devolve estruturas grandes. Nesta etapa nós extraímos apenas os landmarks necessários para o protótipo e os transformamos em um formato pequeno e previsível. Isso deixa as próximas células mais simples de entender.


In [5]:
@dataclass(frozen=True)
# Representa um landmark unico com coordenadas e metadados minimos.
class PointData:
    x: float
    y: float
    z: float | None = None
    visibility: float | None = None
    space: str = 'image'


POSE_LANDMARK_INDEX = {
    'nose': 0,
    'left_ear': 7,
    'right_ear': 8,
    'mouth_left': 9,
    'mouth_right': 10,
    'left_shoulder': 11,
    'right_shoulder': 12,
}

HAND_LANDMARK_INDEX = {
    'wrist': 0,
    'thumb_tip': 4,
    'index_tip': 8,
    'middle_tip': 12,
}

FACE_ANCHOR_NAMES = ['nose', 'mouth_left', 'mouth_right', 'left_ear', 'right_ear']
LEFT_HAND_NAMES = ['wrist', 'index_tip', 'middle_tip', 'thumb_tip']
RIGHT_HAND_NAMES = ['wrist', 'index_tip', 'middle_tip', 'thumb_tip']


# Converte o objeto bruto do MediaPipe em um PointData padronizado.
def make_point(landmark: Any, *, space: str, visibility: float | None = None) -> PointData:
    return PointData(
        x=float(landmark.x),
        y=float(landmark.y),
        z=float(getattr(landmark, 'z', 0.0)),
        visibility=visibility,
        space=space,
    )


# Extrai apenas os landmarks de interesse e filtra pontos invalidos.
def landmark_list_to_points(
    landmark_list: Any,
    name_to_index: Mapping[str, int],
    *,
    space: str,
    require_visibility: bool = False,
    visibility_threshold: float = VISIBILITY_THRESHOLD,
) -> dict[str, PointData]:
    if landmark_list is None:
        return {}

    if hasattr(landmark_list, 'landmark'):
        landmarks = getattr(landmark_list, 'landmark', [])
    else:
        landmarks = landmark_list or []

    points: dict[str, PointData] = {}
    for name, index in name_to_index.items():
        if index >= len(landmarks):
            continue
        landmark = landmarks[index]
        visibility = getattr(landmark, 'visibility', None)
        if require_visibility and (visibility is None or visibility < visibility_threshold):
            continue
        points[name] = make_point(landmark, space=space, visibility=visibility)
    return points


# Escolhe o rotulo de handedness mais provavel devolvido pelo HandLandmarker.
def choose_handed_label(categories: Iterable[Any] | None) -> str | None:
    if not categories:
        return None
    best_category = max(categories, key=lambda category: float(category.score or 0.0))
    label = str(best_category.category_name or best_category.display_name or '').strip().lower()
    if label in {'left', 'right'}:
        return label
    return None


# Seleciona e organiza as maos detectadas em esquerda e direita a partir do resultado da task.
def extract_hand_landmark_sets(hand_result: Any) -> dict[str, dict[str, PointData]]:
    left_hand_points: dict[str, PointData] = {}
    right_hand_points: dict[str, PointData] = {}
    unknown_hands: list[dict[str, PointData]] = []

    if hand_result is None:
        return {
            'left_hand_2d': left_hand_points,
            'right_hand_2d': right_hand_points,
        }

    hand_landmarks = getattr(hand_result, 'hand_landmarks', []) or []
    handedness_lists = getattr(hand_result, 'handedness', []) or []

    for hand_idx, landmark_list in enumerate(hand_landmarks):
        hand_points = landmark_list_to_points(
            landmark_list,
            HAND_LANDMARK_INDEX,
            space='image',
            require_visibility=False,
        )
        if not hand_points:
            continue

        handedness = handedness_lists[hand_idx] if hand_idx < len(handedness_lists) else None
        hand_label = choose_handed_label(handedness)

        if hand_label == 'left':
            if len(hand_points) >= len(left_hand_points):
                left_hand_points = hand_points
        elif hand_label == 'right':
            if len(hand_points) >= len(right_hand_points):
                right_hand_points = hand_points
        else:
            unknown_hands.append(hand_points)

    for hand_points in unknown_hands:
        if not left_hand_points:
            left_hand_points = hand_points
        elif not right_hand_points:
            right_hand_points = hand_points

    return {
        'left_hand_2d': left_hand_points,
        'right_hand_2d': right_hand_points,
    }


# Organiza pose e maos em grupos separados para o restante do pipeline.
def extract_landmark_sets(pose_result: Any, hand_result: Any) -> dict[str, dict[str, PointData]]:
    pose_landmarks = None
    pose_world_landmarks = None

    if pose_result is not None:
        pose_candidates = getattr(pose_result, 'pose_landmarks', []) or []
        world_candidates = getattr(pose_result, 'pose_world_landmarks', []) or []
        if pose_candidates:
            pose_landmarks = pose_candidates[0]
        if world_candidates:
            pose_world_landmarks = world_candidates[0]

    hand_sets = extract_hand_landmark_sets(hand_result)
    return {
        'pose_2d': landmark_list_to_points(
            pose_landmarks,
            POSE_LANDMARK_INDEX,
            space='image',
            require_visibility=True,
        ),
        'pose_world': landmark_list_to_points(
            pose_world_landmarks,
            POSE_LANDMARK_INDEX,
            space='world',
            require_visibility=False,
        ),
        'left_hand_2d': hand_sets['left_hand_2d'],
        'right_hand_2d': hand_sets['right_hand_2d'],
    }


# Devolve uma estrutura vazia quando a inferencia do frame falha totalmente.
def build_empty_landmark_sets() -> dict[str, dict[str, PointData]]:
    return {
        'pose_2d': {},
        'pose_world': {},
        'left_hand_2d': {},
        'right_hand_2d': {},
    }


# Copia apenas os pontos 2D que serao usados no overlay do video.
def clone_drawing_points(landmark_sets: Mapping[str, dict[str, PointData]]) -> dict[str, dict[str, PointData]]:
    return {
        'pose_2d': dict(landmark_sets.get('pose_2d', {})),
        'left_hand_2d': dict(landmark_sets.get('left_hand_2d', {})),
        'right_hand_2d': dict(landmark_sets.get('right_hand_2d', {})),
    }


## 6) Suavização temporal com EMA

Landmarks variam bastante de frame para frame. A média móvel exponencial (EMA) reduz esse ruído sem apagar mudanças rápidas. O detalhe importante aqui é que cada landmark é suavizado de forma independente, para que o desaparecimento de uma mão não contamine os demais pontos.


In [6]:
# Faz a mistura EMA quando um valor pode estar ausente em um dos frames.
def blend_optional(current_value: float | None, previous_value: float | None, alpha: float) -> float | None:
    if current_value is None and previous_value is None:
        return None
    if current_value is None:
        return previous_value
    if previous_value is None:
        return current_value
    return (alpha * current_value) + ((1.0 - alpha) * previous_value)


# Mantém o histórico por landmark e aplica suavização temporal frame a frame.
class EMASmoother:
    # Inicializa o fator da EMA e o limite de frames para resetar o historico.
    def __init__(self, alpha: float = EMA_ALPHA, max_gap_frames: int = EMA_MAX_GAP_FRAMES) -> None:
        self.alpha = alpha
        self.max_gap_frames = max_gap_frames
        self.state: dict[str, dict[str, Any]] = {}

    # Atualiza os landmarks do frame atual e devolve a versao suavizada.
    def update(self, frame_idx: int, points: Mapping[str, PointData]) -> dict[str, PointData]:
        smoothed: dict[str, PointData] = {}
        for name, point in points.items():
            previous = self.state.get(name)

            # Resetamos a EMA quando o ponto some por muitos frames. Isso evita misturar
            # posições antigas com um reaparecimento em local muito diferente.
            if previous is None or (frame_idx - previous['frame_idx']) > self.max_gap_frames:
                smoothed_point = point
            else:
                previous_point = previous['point']
                smoothed_point = PointData(
                    x=(self.alpha * point.x) + ((1.0 - self.alpha) * previous_point.x),
                    y=(self.alpha * point.y) + ((1.0 - self.alpha) * previous_point.y),
                    z=blend_optional(point.z, previous_point.z, self.alpha),
                    visibility=point.visibility,
                    space=point.space,
                )

            self.state[name] = {'frame_idx': frame_idx, 'point': smoothed_point}
            smoothed[name] = smoothed_point
        return smoothed


# Cria um suavizador independente para cada grupo de landmarks usado no pipeline.
def build_smoothers() -> dict[str, EMASmoother]:
    return {
        'pose_2d': EMASmoother(),
        'pose_world': EMASmoother(),
        'left_hand_2d': EMASmoother(),
        'right_hand_2d': EMASmoother(),
    }


# Aplica os suavizadores corretos a todos os conjuntos de landmarks extraidos no frame.
def smooth_landmark_sets(
    frame_idx: int,
    raw_sets: Mapping[str, dict[str, PointData]],
    smoothers: Mapping[str, EMASmoother],
) -> dict[str, dict[str, PointData]]:
    return {
        name: smoothers[name].update(frame_idx, raw_sets.get(name, {}))
        for name in ['pose_2d', 'pose_world', 'left_hand_2d', 'right_hand_2d']
    }


## 7) Geometria de referência

Antes de avaliar regras, precisamos construir referências estáveis: centro dos ombros, largura dos ombros, pontos do rosto, pontos das mãos e medidas normalizadas. Normalizar por largura dos ombros torna o protótipo mais robusto a distância da câmera e resolução do vídeo.


In [7]:
# Arredonda valores numericos opcionais sem quebrar quando o dado estiver ausente.
def optional_round(value: float | None, digits: int = 4) -> float | None:
    if value is None:
        return None
    return round(float(value), digits)


# Calcula a distancia entre dois pontos usando 2D ou 3D quando disponivel.
def point_distance(point_a: PointData | None, point_b: PointData | None) -> float | None:
    if point_a is None or point_b is None:
        return None
    if point_a.space == point_b.space == 'world' and point_a.z is not None and point_b.z is not None:
        return math.dist((point_a.x, point_a.y, point_a.z), (point_b.x, point_b.y, point_b.z))
    return math.dist((point_a.x, point_a.y), (point_b.x, point_b.y))


# Calcula o ponto medio entre dois landmarks compativeis.
def midpoint(point_a: PointData | None, point_b: PointData | None) -> PointData | None:
    if point_a is None or point_b is None:
        return None
    z_value = None
    if point_a.z is not None and point_b.z is not None:
        z_value = (point_a.z + point_b.z) / 2.0
    visibility = None
    if point_a.visibility is not None and point_b.visibility is not None:
        visibility = min(point_a.visibility, point_b.visibility)
    return PointData(
        x=(point_a.x + point_b.x) / 2.0,
        y=(point_a.y + point_b.y) / 2.0,
        z=z_value,
        visibility=visibility,
        space=point_a.space,
    )


# Desloca um ponto de referencia no eixo vertical para criar centros auxiliares.
def offset_point(point: PointData | None, dy: float) -> PointData | None:
    if point is None:
        return None
    return PointData(x=point.x, y=point.y + dy, z=point.z, visibility=point.visibility, space=point.space)


# Remove pontos ausentes e devolve apenas landmarks validos.
def valid_points(points: Iterable[PointData | None]) -> list[PointData]:
    return [point for point in points if point is not None]


# Encontra a menor distancia entre dois conjuntos de pontos.
def min_distance_between_sets(points_a: Iterable[PointData], points_b: Iterable[PointData]) -> float | None:
    distances = [
        point_distance(point_a, point_b)
        for point_a in points_a
        for point_b in points_b
        if point_distance(point_a, point_b) is not None
    ]
    return min(distances) if distances else None


# Mede a menor distancia entre um conjunto de pontos e um centro de referencia.
def min_distance_to_center(points: Iterable[PointData], center_point: PointData | None) -> float | None:
    if center_point is None:
        return None
    distances = [point_distance(point, center_point) for point in points]
    distances = [distance for distance in distances if distance is not None]
    return min(distances) if distances else None


# Monta todas as referencias geometricas usadas pelas regras do prototipo.
def compute_reference_geometry(landmark_sets: Mapping[str, dict[str, PointData]]) -> dict[str, Any]:
    pose_2d = landmark_sets.get('pose_2d', {})
    pose_world = landmark_sets.get('pose_world', {})
    left_hand_2d = landmark_sets.get('left_hand_2d', {})
    right_hand_2d = landmark_sets.get('right_hand_2d', {})

    left_shoulder = pose_2d.get('left_shoulder')
    right_shoulder = pose_2d.get('right_shoulder')
    shoulder_mid = midpoint(left_shoulder, right_shoulder)
    shoulder_width = point_distance(left_shoulder, right_shoulder)
    shoulder_reference_ok = shoulder_width is not None and shoulder_width >= MIN_SHOULDER_WIDTH_NORM

    face_points = valid_points([pose_2d.get(name) for name in FACE_ANCHOR_NAMES])
    left_hand_points = valid_points([left_hand_2d.get(name) for name in LEFT_HAND_NAMES])
    right_hand_points = valid_points([right_hand_2d.get(name) for name in RIGHT_HAND_NAMES])
    all_hand_points = left_hand_points + right_hand_points

    world_left_shoulder = pose_world.get('left_shoulder')
    world_right_shoulder = pose_world.get('right_shoulder')
    shoulder_mid_world = midpoint(world_left_shoulder, world_right_shoulder)
    shoulder_width_world = point_distance(world_left_shoulder, world_right_shoulder)
    nose_world = pose_world.get('nose')

    neck_center = None
    upper_chest_center = None
    head_height_ratio = None
    if shoulder_reference_ok and shoulder_mid is not None:
        neck_center = offset_point(shoulder_mid, dy=(-0.12 * shoulder_width))
        upper_chest_center = offset_point(shoulder_mid, dy=(0.18 * shoulder_width))
        nose_2d = pose_2d.get('nose')
        if nose_2d is not None:
            head_height_ratio = (shoulder_mid.y - nose_2d.y) / shoulder_width

    left_shoulder_ear_ratio = None
    right_shoulder_ear_ratio = None
    if shoulder_reference_ok:
        left_ear = pose_2d.get('left_ear')
        right_ear = pose_2d.get('right_ear')
        left_shoulder_ear = point_distance(left_shoulder, left_ear)
        right_shoulder_ear = point_distance(right_shoulder, right_ear)
        if left_shoulder_ear is not None:
            left_shoulder_ear_ratio = left_shoulder_ear / shoulder_width
        if right_shoulder_ear is not None:
            right_shoulder_ear_ratio = right_shoulder_ear / shoulder_width

    shoulder_asymmetry_ratio = None
    if shoulder_reference_ok and left_shoulder is not None and right_shoulder is not None:
        shoulder_asymmetry_ratio = abs(left_shoulder.y - right_shoulder.y) / shoulder_width

    forward_head_world_ratio = None
    if shoulder_mid_world is not None and shoulder_width_world and shoulder_width_world > 0 and nose_world is not None:
        forward_head_world_ratio = (shoulder_mid_world.z - nose_world.z) / shoulder_width_world

    coordinate_mode = COORDINATE_MODE_WORLD if forward_head_world_ratio is not None else COORDINATE_MODE_2D

    return {
        'shoulder_mid': shoulder_mid,
        'shoulder_width': shoulder_width,
        'shoulder_reference_ok': shoulder_reference_ok,
        'face_points': face_points,
        'left_hand_points': left_hand_points,
        'right_hand_points': right_hand_points,
        'all_hand_points': all_hand_points,
        'neck_center': neck_center,
        'upper_chest_center': upper_chest_center,
        'head_height_ratio': head_height_ratio,
        'left_shoulder_ear_ratio': left_shoulder_ear_ratio,
        'right_shoulder_ear_ratio': right_shoulder_ear_ratio,
        'shoulder_asymmetry_ratio': shoulder_asymmetry_ratio,
        'forward_head_world_ratio': forward_head_world_ratio,
        'coordinate_mode': coordinate_mode,
        'hands_visible': bool(all_hand_points),
        'left_hand_visible': bool(left_hand_points),
        'right_hand_visible': bool(right_hand_points),
        'face_anchor_count': len(face_points),
        'pose_2d': pose_2d,
        'pose_world': pose_world,
    }

## 8) Regras de postura e self-touch

Agora aplicamos as seis regras do plano. Cada regra devolve `state`, `strength` e métricas de apoio. Quando faltam landmarks, a regra vira `unknown`; o notebook não transforma ausência de evidência em `false`.


In [8]:
# Padroniza a saida de uma regra com estado, forca e metricas.
def build_rule_result(state: str, strength: str, **metrics: float | str | None) -> dict[str, Any]:
    serialized_metrics: dict[str, Any] = {}
    for key, value in metrics.items():
        if isinstance(value, float):
            serialized_metrics[key] = optional_round(value)
        else:
            serialized_metrics[key] = value
    return {'state': state, 'strength': strength, 'metrics': serialized_metrics}


# Calcula a melhor distancia normalizada entre maos e um conjunto de alvos.
def best_hand_ratio_to_targets(
    geometry: Mapping[str, Any],
    target_points: Iterable[PointData],
) -> tuple[float | None, str | None]:
    shoulder_width = geometry.get('shoulder_width')
    if not geometry.get('shoulder_reference_ok') or not shoulder_width:
        return None, None

    candidates: list[tuple[float, str]] = []
    left_distance = min_distance_between_sets(geometry.get('left_hand_points', []), list(target_points))
    if left_distance is not None:
        candidates.append((left_distance / shoulder_width, 'left'))

    right_distance = min_distance_between_sets(geometry.get('right_hand_points', []), list(target_points))
    if right_distance is not None:
        candidates.append((right_distance / shoulder_width, 'right'))

    if not candidates:
        return None, None
    return min(candidates, key=lambda item: item[0])


# Calcula a melhor distancia normalizada entre maos e um ponto central.
def best_hand_ratio_to_center(geometry: Mapping[str, Any], center_point: PointData | None) -> tuple[float | None, str | None]:
    shoulder_width = geometry.get('shoulder_width')
    if not geometry.get('shoulder_reference_ok') or not shoulder_width:
        return None, None

    candidates: list[tuple[float, str]] = []
    left_distance = min_distance_to_center(geometry.get('left_hand_points', []), center_point)
    if left_distance is not None:
        candidates.append((left_distance / shoulder_width, 'left'))

    right_distance = min_distance_to_center(geometry.get('right_hand_points', []), center_point)
    if right_distance is not None:
        candidates.append((right_distance / shoulder_width, 'right'))

    if not candidates:
        return None, None
    return min(candidates, key=lambda item: item[0])


# Avalia a regra de contato da mao com o rosto.
def evaluate_hand_on_face(geometry: Mapping[str, Any]) -> dict[str, Any]:
    ratio, hand_side = best_hand_ratio_to_targets(geometry, geometry.get('face_points', []))
    if ratio is None:
        return build_rule_result('unknown', 'none', min_face_hand_ratio=None, closest_hand=None)
    if ratio < 0.25:
        return build_rule_result('true', 'strong', min_face_hand_ratio=ratio, closest_hand=hand_side)
    if ratio < 0.30:
        return build_rule_result('true', 'weak', min_face_hand_ratio=ratio, closest_hand=hand_side)
    return build_rule_result('false', 'none', min_face_hand_ratio=ratio, closest_hand=hand_side)


# Avalia a regra de contato da mao com o pescoco respeitando a prioridade das regras.
def evaluate_hand_on_neck(geometry: Mapping[str, Any], hand_on_face_rule: Mapping[str, Any]) -> dict[str, Any]:
    ratio, hand_side = best_hand_ratio_to_center(geometry, geometry.get('neck_center'))
    if ratio is None:
        return build_rule_result('unknown', 'none', min_neck_hand_ratio=None, closest_hand=None)
    if hand_on_face_rule.get('state') == 'true':
        return build_rule_result('false', 'none', min_neck_hand_ratio=ratio, closest_hand=hand_side, suppressed_by='hand_on_face')
    if ratio < 0.18:
        return build_rule_result('true', 'strong', min_neck_hand_ratio=ratio, closest_hand=hand_side)
    if ratio < 0.22:
        return build_rule_result('true', 'weak', min_neck_hand_ratio=ratio, closest_hand=hand_side)
    return build_rule_result('false', 'none', min_neck_hand_ratio=ratio, closest_hand=hand_side)


# Avalia a regra de contato da mao com o peito apos checar supressoes.
def evaluate_hand_on_chest(
    geometry: Mapping[str, Any],
    hand_on_face_rule: Mapping[str, Any],
    hand_on_neck_rule: Mapping[str, Any],
) -> dict[str, Any]:
    ratio, hand_side = best_hand_ratio_to_center(geometry, geometry.get('upper_chest_center'))
    if ratio is None:
        return build_rule_result('unknown', 'none', min_chest_hand_ratio=None, closest_hand=None)
    if hand_on_face_rule.get('state') == 'true':
        return build_rule_result('false', 'none', min_chest_hand_ratio=ratio, closest_hand=hand_side, suppressed_by='hand_on_face')
    if hand_on_neck_rule.get('state') == 'true':
        return build_rule_result('false', 'none', min_chest_hand_ratio=ratio, closest_hand=hand_side, suppressed_by='hand_on_neck')
    if ratio < 0.22:
        return build_rule_result('true', 'strong', min_chest_hand_ratio=ratio, closest_hand=hand_side)
    if ratio < 0.26:
        return build_rule_result('true', 'weak', min_chest_hand_ratio=ratio, closest_hand=hand_side)
    return build_rule_result('false', 'none', min_chest_hand_ratio=ratio, closest_hand=hand_side)


# Avalia a inclinacao da cabeca usando a altura relativa do nariz.
def evaluate_head_down(geometry: Mapping[str, Any]) -> dict[str, Any]:
    ratio = geometry.get('head_height_ratio')
    if ratio is None:
        return build_rule_result('unknown', 'none', head_height_ratio=None)
    if ratio < 0.30:
        return build_rule_result('true', 'strong', head_height_ratio=ratio)
    if ratio < 0.40:
        return build_rule_result('true', 'weak', head_height_ratio=ratio)
    return build_rule_result('false', 'none', head_height_ratio=ratio)


# Avalia cabeca projetada para frente usando 3D ou fallback 2D conservador.
def evaluate_forward_head(geometry: Mapping[str, Any], head_down_rule: Mapping[str, Any]) -> dict[str, Any]:
    world_ratio = geometry.get('forward_head_world_ratio')
    if world_ratio is not None:
        if world_ratio > 0.16:
            return build_rule_result('true', 'strong', forward_head_ratio=world_ratio, basis='world')
        if world_ratio > 0.10:
            return build_rule_result('true', 'weak', forward_head_ratio=world_ratio, basis='world')
        return build_rule_result('false', 'none', forward_head_ratio=world_ratio, basis='world')

    # O fallback 2D nunca devolve evidência forte sozinho. Ele combina pistas visuais.
    head_down_active = head_down_rule.get('state') == 'true'
    left_ratio = geometry.get('left_shoulder_ear_ratio')
    right_ratio = geometry.get('right_shoulder_ear_ratio')
    available_ear_ratios = [ratio for ratio in [left_ratio, right_ratio] if ratio is not None]
    mean_shoulder_ear_ratio = float(np.mean(available_ear_ratios)) if available_ear_ratios else None
    head_height_ratio = geometry.get('head_height_ratio')

    evidence_votes = 0
    evidence_capacity = 0

    evidence_capacity += 1
    if head_down_active:
        evidence_votes += 1

    if mean_shoulder_ear_ratio is not None:
        evidence_capacity += 1
        if mean_shoulder_ear_ratio < 0.31:
            evidence_votes += 1

    if head_height_ratio is not None:
        evidence_capacity += 1
        if head_height_ratio < 0.36:
            evidence_votes += 1

    if evidence_capacity < 2:
        return build_rule_result(
            'unknown',
            'none',
            forward_head_ratio=None,
            basis='weak_2d_fallback',
            evidence_votes=evidence_votes,
            evidence_capacity=evidence_capacity,
        )
    if evidence_votes >= 2:
        return build_rule_result(
            'true',
            'weak',
            forward_head_ratio=None,
            basis='weak_2d_fallback',
            mean_shoulder_ear_ratio=mean_shoulder_ear_ratio,
            head_height_ratio=head_height_ratio,
            evidence_votes=evidence_votes,
            evidence_capacity=evidence_capacity,
        )
    return build_rule_result(
        'false',
        'none',
        forward_head_ratio=None,
        basis='weak_2d_fallback',
        mean_shoulder_ear_ratio=mean_shoulder_ear_ratio,
        head_height_ratio=head_height_ratio,
        evidence_votes=evidence_votes,
        evidence_capacity=evidence_capacity,
    )


# Avalia tensao de ombros pela combinacao de elevacao e assimetria.
def evaluate_rounded_shoulders_or_asymmetry(geometry: Mapping[str, Any]) -> dict[str, Any]:
    left_ratio = geometry.get('left_shoulder_ear_ratio')
    right_ratio = geometry.get('right_shoulder_ear_ratio')
    asymmetry_ratio = geometry.get('shoulder_asymmetry_ratio')

    shrugged_shoulders = (
        left_ratio is not None and right_ratio is not None and left_ratio < 0.33 and right_ratio < 0.33
    )
    shoulder_asymmetry = asymmetry_ratio is not None and asymmetry_ratio > 0.12
    evaluable = any(value is not None for value in [left_ratio, right_ratio, asymmetry_ratio])

    if not evaluable:
        return build_rule_result(
            'unknown',
            'none',
            shrugged_shoulders=None,
            shoulder_asymmetry=None,
            left_shoulder_ear_ratio=None,
            right_shoulder_ear_ratio=None,
            shoulder_asymmetry_ratio=None,
        )

    if shrugged_shoulders or shoulder_asymmetry:
        strength = 'strong' if (shrugged_shoulders and shoulder_asymmetry) or (asymmetry_ratio is not None and asymmetry_ratio > 0.18) else 'weak'
        return build_rule_result(
            'true',
            strength,
            shrugged_shoulders=shrugged_shoulders,
            shoulder_asymmetry=shoulder_asymmetry,
            left_shoulder_ear_ratio=left_ratio,
            right_shoulder_ear_ratio=right_ratio,
            shoulder_asymmetry_ratio=asymmetry_ratio,
        )

    return build_rule_result(
        'false',
        'none',
        shrugged_shoulders=shrugged_shoulders,
        shoulder_asymmetry=shoulder_asymmetry,
        left_shoulder_ear_ratio=left_ratio,
        right_shoulder_ear_ratio=right_ratio,
        shoulder_asymmetry_ratio=asymmetry_ratio,
    )


# Executa todas as regras do frame e devolve o conjunto consolidado.
def evaluate_rules(geometry: Mapping[str, Any]) -> dict[str, dict[str, Any]]:
    hand_on_face = evaluate_hand_on_face(geometry)
    hand_on_neck = evaluate_hand_on_neck(geometry, hand_on_face)
    hand_on_chest = evaluate_hand_on_chest(geometry, hand_on_face, hand_on_neck)
    head_down = evaluate_head_down(geometry)
    forward_head = evaluate_forward_head(geometry, head_down)
    rounded_shoulders_or_asymmetry = evaluate_rounded_shoulders_or_asymmetry(geometry)
    return {
        'hand_on_face': hand_on_face,
        'hand_on_neck': hand_on_neck,
        'hand_on_chest': hand_on_chest,
        'head_down': head_down,
        'forward_head': forward_head,
        'rounded_shoulders_or_asymmetry': rounded_shoulders_or_asymmetry,
    }


## 9) Scoring por frame, janelas deslizantes e serialização

Com as regras prontas, esta etapa transforma sinais pontuais em resumos por frame, por janela e por vídeo inteiro. Também é aqui que montamos as limitações explícitas e o payload JSON final.

Pose em vídeo oscila muito: mão sai do quadro, landmark falha, cabeça mexe por um instante. Se você decidir só por frame, o resultado fica ruidoso e reativo demais. A janela deslizante suaviza isso porque olha um trecho inteiro e pergunta: “esse padrão apareceu de forma recorrente aqui ou foi só um evento isolado?”

Na prática, ela ajuda em três coisas:
* reduz falsos positivos causados por um frame ruim ou um movimento curto;
* captura persistência, que é mais relevante para postura do que um instante solto;
* produz um resumo temporal mais interpretável, porque você passa a ter “neste intervalo houve sinal” em vez de milhares de decisões fragmentadas.

In [9]:
# Traduz o nome interno da regra para um rotulo mais legivel.
def pretty_rule_name(rule_name: str | None) -> str | None:
    if rule_name is None:
        return None
    return RULE_DISPLAY_NAMES.get(rule_name, rule_name.replace('_', ' '))


# Verifica se uma regra foi ativada no frame atual.
def rule_is_true(rule: Mapping[str, Any]) -> bool:
    return rule.get('state') == 'true'


# Escolhe a regra ativa mais relevante com base no peso configurado.
def choose_primary_trigger(rules: Mapping[str, Mapping[str, Any]]) -> str | None:
    active_rules = [name for name, rule in rules.items() if rule_is_true(rule)]
    if not active_rules:
        return None
    return max(active_rules, key=lambda name: RULE_WEIGHTS.get(name, 0.0))


# Gera uma explicacao curta em English para o resultado do frame.
def build_frame_explanation(rules: Mapping[str, Mapping[str, Any]], frame_level: str) -> str:
    if frame_level == 'insufficient_data':
        return 'insufficient data'
    if rule_is_true(rules['hand_on_face']) and rule_is_true(rules['head_down']):
        return 'recurrent hand-to-face contact + head down'
    if rule_is_true(rules['hand_on_neck']) and rule_is_true(rules['forward_head']):
        return 'recurrent hand-to-neck contact + forward head'
    if rule_is_true(rules['rounded_shoulders_or_asymmetry']):
        return 'persistent upper-body tension / asymmetry'
    for rule_name in ['hand_on_face', 'hand_on_neck', 'hand_on_chest', 'head_down', 'forward_head']:
        if rule_is_true(rules[rule_name]):
            return pretty_rule_name(rule_name) or 'observable posture signal'
    return 'no major posture signal'


# Consolida score, nivel, metricas e explicacao de um frame amostrado.
def build_frame_signal_record(
    frame_idx: int,
    timestamp_s: float,
    geometry: Mapping[str, Any],
    rules: Mapping[str, Mapping[str, Any]],
) -> dict[str, Any]:
    active_rules = [name for name, rule in rules.items() if rule.get('state') == 'true']
    known_rules = [name for name, rule in rules.items() if rule.get('state') != 'unknown']
    unknown_rules = [name for name, rule in rules.items() if rule.get('state') == 'unknown']
    frame_score = sum(RULE_WEIGHTS.get(name, 0.0) for name in active_rules)

    partial_data = bool(known_rules) and bool(unknown_rules)
    insufficient_data = (not geometry.get('shoulder_reference_ok')) or not known_rules

    if insufficient_data:
        frame_level = 'insufficient_data'
    elif (rule_is_true(rules['hand_on_face']) and rule_is_true(rules['head_down'])) or (
        rule_is_true(rules['hand_on_neck']) and rule_is_true(rules['forward_head'])
    ) or frame_score >= 2.4:
        frame_level = 'strong_signal'
    elif frame_score >= 1.8:
        frame_level = 'possible_signal'
    else:
        frame_level = 'ok'

    explanation = build_frame_explanation(rules, frame_level)
    primary_trigger = choose_primary_trigger(rules)

    metrics = {
        'shoulder_width': optional_round(geometry.get('shoulder_width')),
        'head_height_ratio': optional_round(geometry.get('head_height_ratio')),
        'forward_head_world_ratio': optional_round(geometry.get('forward_head_world_ratio')),
        'left_shoulder_ear_ratio': optional_round(geometry.get('left_shoulder_ear_ratio')),
        'right_shoulder_ear_ratio': optional_round(geometry.get('right_shoulder_ear_ratio')),
        'shoulder_asymmetry_ratio': optional_round(geometry.get('shoulder_asymmetry_ratio')),
        'face_anchor_count': int(geometry.get('face_anchor_count', 0)),
        'left_hand_visible': bool(geometry.get('left_hand_visible')),
        'right_hand_visible': bool(geometry.get('right_hand_visible')),
        'coordinate_mode': geometry.get('coordinate_mode'),
    }

    return {
        'frame_idx': int(frame_idx),
        'timestamp_s': round(float(timestamp_s), 3),
        'status': 'insufficient_data' if insufficient_data else 'scorable',
        'partial_data': partial_data,
        'metrics': metrics,
        'rules': rules,
        'frame_score': round(float(frame_score), 3),
        'frame_level': frame_level,
        'explanation': explanation,
        'primary_trigger': primary_trigger,
        'active_rules': active_rules,
    }


# Gera os intervalos de janelas deslizantes ao longo da duracao do video.
def window_ranges(duration_seconds: float, window_seconds: float, stride_seconds: float) -> list[tuple[int, float, float]]:
    if duration_seconds <= 0:
        return []
    if duration_seconds <= window_seconds:
        return [(0, 0.0, duration_seconds)]

    ranges: list[tuple[int, float, float]] = []
    window_idx = 0
    start_s = 0.0
    while start_s < duration_seconds:
        end_s = min(duration_seconds, start_s + window_seconds)
        ranges.append((window_idx, start_s, end_s))
        if end_s >= duration_seconds:
            break
        window_idx += 1
        start_s += stride_seconds
    return ranges


# Conta a maior sequencia seguida de frames com sinal.
def count_max_consecutive_signal_frames(frame_records: Iterable[Mapping[str, Any]]) -> int:
    best_run = 0
    current_run = 0
    for frame in frame_records:
        if frame.get('frame_level') in {'possible_signal', 'strong_signal'}:
            current_run += 1
            best_run = max(best_run, current_run)
        else:
            current_run = 0
    return best_run


# Converte a razao de sinais de uma janela em um nivel discreto.
def label_window(signal_ratio: float | None) -> str:
    if signal_ratio is None:
        return 'insufficient_data'
    if signal_ratio > 0.35:
        return 'strong_signal'
    if signal_ratio >= 0.15:
        return 'possible_signal'
    return 'ok'


# Resume os frames em metricas por janela deslizante.
def aggregate_windows(frame_records: list[dict[str, Any]], duration_seconds: float) -> list[dict[str, Any]]:
    summaries: list[dict[str, Any]] = []
    for window_idx, start_s, end_s in window_ranges(duration_seconds, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS):
        window_frames = [
            frame
            for frame in frame_records
            if start_s <= frame.get('timestamp_s', 0.0) < (end_s + 1e-9)
        ]

        scorable_frames = [frame for frame in window_frames if frame.get('frame_level') != 'insufficient_data']
        frames_total = len(window_frames)
        frames_scorable = len(scorable_frames)
        frames_insufficient = frames_total - frames_scorable

        signal_frames = [
            frame for frame in scorable_frames if frame.get('frame_level') in {'possible_signal', 'strong_signal'}
        ]
        strong_frames = [frame for frame in scorable_frames if frame.get('frame_level') == 'strong_signal']
        signal_ratio = (len(signal_frames) / frames_scorable) if frames_scorable else None
        strong_signal_ratio = (len(strong_frames) / frames_scorable) if frames_scorable else None
        frame_scores = [float(frame.get('frame_score', 0.0)) for frame in scorable_frames]
        trigger_counter = Counter(
            frame.get('primary_trigger') for frame in signal_frames if frame.get('primary_trigger') is not None
        )
        most_common_trigger = trigger_counter.most_common(1)[0][0] if trigger_counter else None
        window_level = label_window(signal_ratio)

        if window_level == 'insufficient_data':
            explanation = 'insufficient data coverage'
        elif most_common_trigger is not None:
            explanation = f"{pretty_rule_name(most_common_trigger)} recurring across the window"
        else:
            explanation = 'no major posture signal'

        summaries.append({
            'window_idx': window_idx,
            'start_s': round(start_s, 3),
            'end_s': round(end_s, 3),
            'frames_total': frames_total,
            'frames_scorable': frames_scorable,
            'frames_insufficient': frames_insufficient,
            'signal_ratio': optional_round(signal_ratio),
            'strong_signal_ratio': optional_round(strong_signal_ratio),
            'max_consecutive_signal_frames': count_max_consecutive_signal_frames(window_frames),
            'most_common_trigger': most_common_trigger,
            'window_score_mean': optional_round(float(np.mean(frame_scores)) if frame_scores else None),
            'window_score_peak': optional_round(float(np.max(frame_scores)) if frame_scores else None),
            'window_level': window_level,
            'explanation': explanation,
        })
    return summaries


# Retorna o nivel mais severo observado entre as janelas.
def peak_window_level(window_summaries: Iterable[Mapping[str, Any]]) -> str:
    best_level = 'insufficient_data'
    best_score = WINDOW_LEVEL_SEVERITY[best_level]
    for summary in window_summaries:
        level = summary.get('window_level', 'insufficient_data')
        if WINDOW_LEVEL_SEVERITY.get(level, -1) > best_score:
            best_level = level
            best_score = WINDOW_LEVEL_SEVERITY[level]
    return best_level


# Detecta janelas fortes consecutivas para a regra final do video.
def has_adjacent_strong_windows(valid_windows: list[Mapping[str, Any]]) -> bool:
    run_length = 0
    previous_index = None
    for window in valid_windows:
        if window.get('window_level') == 'strong_signal':
            if previous_index is not None and window.get('window_idx') == previous_index + 1:
                run_length += 1
            else:
                run_length = 1
            if run_length >= 2:
                return True
        else:
            run_length = 0
        previous_index = window.get('window_idx')
    return False


# Agrega as janelas validas em um resumo final do video.
def build_video_summary(frame_records: list[dict[str, Any]], window_summaries: list[dict[str, Any]]) -> dict[str, Any]:
    sampled_frames = len(frame_records)
    scorable_frames = sum(frame.get('status') == 'scorable' for frame in frame_records)
    coverage_ratio = (scorable_frames / sampled_frames) if sampled_frames else 0.0

    valid_windows = [window for window in window_summaries if window.get('window_level') != 'insufficient_data']
    trigger_counter = Counter(
        window.get('most_common_trigger') for window in valid_windows if window.get('most_common_trigger') is not None
    )
    most_common_trigger = trigger_counter.most_common(1)[0][0] if trigger_counter else None

    if not valid_windows:
        return {
            'level': 'insufficient_data',
            'video_signal_ratio': None,
            'video_strong_window_ratio': None,
            'coverage_ratio': optional_round(coverage_ratio),
            'peak_window_level': 'insufficient_data',
            'most_common_trigger': most_common_trigger,
            'dominant_explanation': 'insufficient upper-body coverage for reliable screening',
            'explanation': 'insufficient upper-body coverage for reliable screening',
        }

    strong_windows = [window for window in valid_windows if window.get('window_level') == 'strong_signal']
    signal_windows = [
        window for window in valid_windows if window.get('window_level') in {'possible_signal', 'strong_signal'}
    ]

    video_signal_ratio = len(signal_windows) / len(valid_windows)
    video_strong_window_ratio = len(strong_windows) / len(valid_windows)
    if (video_strong_window_ratio > 0.35) or has_adjacent_strong_windows(valid_windows):
        level = 'strong_signal'
    elif video_signal_ratio >= 0.15:
        level = 'possible_signal'
    else:
        level = 'ok'

    if coverage_ratio < 0.10:
        level = 'insufficient_data'
        dominant_explanation = 'insufficient upper-body coverage for reliable screening'
    elif most_common_trigger is not None:
        dominant_explanation = build_frame_explanation(
            {name: {'state': 'true'} if name == most_common_trigger else {'state': 'false'} for name in RULE_WEIGHTS},
            level,
        )
    else:
        dominant_explanation = 'no major posture signal'

    return {
        'level': level,
        'video_signal_ratio': optional_round(video_signal_ratio),
        'video_strong_window_ratio': optional_round(video_strong_window_ratio),
        'coverage_ratio': optional_round(coverage_ratio),
        'peak_window_level': peak_window_level(valid_windows),
        'most_common_trigger': most_common_trigger,
        'dominant_explanation': dominant_explanation,
        'explanation': dominant_explanation,
    }


# Calcula porcentagem simples com protecao para divisao por zero.
def percentage(count: int, total: int) -> float:
    if total <= 0:
        return 0.0
    return round((count / total) * 100.0, 1)


# Monta a lista explicita de limitacoes observadas no processamento.
def build_limitations(frame_records: list[dict[str, Any]]) -> list[str]:
    limitations = [
        'This is a heuristic geometric posture screener based on MediaPipe Holistic landmarks.',
        'The v1 pipeline does not reason about multiple visible people.',
    ]

    total_frames = len(frame_records)
    if total_frames == 0:
        limitations.append('No sampled frames were processed from the source video.')
        return limitations

    insufficient_frames = sum(frame.get('status') == 'insufficient_data' for frame in frame_records)
    partial_frames = sum(bool(frame.get('partial_data')) for frame in frame_records)
    no_hand_frames = sum(
        not frame.get('metrics', {}).get('left_hand_visible') and not frame.get('metrics', {}).get('right_hand_visible')
        for frame in frame_records
    )
    weak_forward_frames = sum(
        frame.get('rules', {}).get('forward_head', {}).get('metrics', {}).get('basis') == 'weak_2d_fallback'
        for frame in frame_records
    )

    if insufficient_frames:
        limitations.append(
            f"Shoulder or face reference was insufficient in {percentage(insufficient_frames, total_frames)}% of sampled frames."
        )
    if partial_frames:
        limitations.append(
            f"At least one rule was unknown in {percentage(partial_frames, total_frames)}% of sampled frames due to partial landmark coverage."
        )
    if no_hand_frames:
        limitations.append(
            f"Hand landmarks were unavailable in {percentage(no_hand_frames, total_frames)}% of sampled frames, which weakens self-touch rules."
        )
    if weak_forward_frames:
        limitations.append(
            f"Forward-head estimation relied on the weak 2D fallback in {percentage(weak_forward_frames, total_frames)}% of sampled frames."
        )
    return limitations


# Normaliza tipos especiais antes de serializar o payload.
def sanitize_for_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {key: sanitize_for_json(item) for key, item in value.items()}
    if isinstance(value, list):
        return [sanitize_for_json(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value


# Salva o payload final em JSON formatado no disco.
def write_payload_json(payload: Mapping[str, Any], json_output_path: Path) -> None:
    json_output_path.parent.mkdir(parents=True, exist_ok=True)
    json_output_path.write_text(
        json.dumps(sanitize_for_json(dict(payload)), ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )


## 10) Renderização do vídeo anotado

O plano pede um segundo passe sobre o vídeo original para gerar o MP4 final. Nesta etapa desenhamos landmarks simplificados, score, regras ativas, janela corrente e explicação curta. Quando há janelas sobrepostas, escolhemos a mais severa e mais recente para o overlay.


In [10]:
POSE_DRAW_SEGMENTS = [
    ('left_ear', 'left_shoulder'),
    ('right_ear', 'right_shoulder'),
    ('left_shoulder', 'right_shoulder'),
    ('nose', 'left_shoulder'),
    ('nose', 'right_shoulder'),
]

HAND_DRAW_SEGMENTS = [
    ('wrist', 'thumb_tip'),
    ('wrist', 'index_tip'),
    ('wrist', 'middle_tip'),
]


# Converte um landmark normalizado em coordenadas de pixel do frame.
def point_to_pixel(point: PointData | None, frame_shape: tuple[int, int, int]) -> tuple[int, int] | None:
    if point is None:
        return None
    height, width = frame_shape[:2]
    x = int(np.clip(point.x, 0.0, 1.0) * width)
    y = int(np.clip(point.y, 0.0, 1.0) * height)
    return x, y


# Desenha uma linha entre dois landmarks quando ambos estao disponiveis.
def draw_segment(frame: np.ndarray, point_a: PointData | None, point_b: PointData | None, color: tuple[int, int, int], thickness: int = 2) -> None:
    pixel_a = point_to_pixel(point_a, frame.shape)
    pixel_b = point_to_pixel(point_b, frame.shape)
    if pixel_a is None or pixel_b is None:
        return
    cv2.line(frame, pixel_a, pixel_b, color, thickness, cv2.LINE_AA)


# Desenha todos os pontos de um conjunto de landmarks no frame.
def draw_points(frame: np.ndarray, points: Mapping[str, PointData], color: tuple[int, int, int], radius: int = 4) -> None:
    for point in points.values():
        pixel = point_to_pixel(point, frame.shape)
        if pixel is None:
            continue
        cv2.circle(frame, pixel, radius, color, -1, cv2.LINE_AA)


# Monta o overlay visual de pose e maos sobre o video anotado.
def draw_landmarks_overlay(frame: np.ndarray, drawing_points: Mapping[str, dict[str, PointData]]) -> None:
    pose_points = drawing_points.get('pose_2d', {})
    left_hand = drawing_points.get('left_hand_2d', {})
    right_hand = drawing_points.get('right_hand_2d', {})

    for start_name, end_name in POSE_DRAW_SEGMENTS:
        draw_segment(frame, pose_points.get(start_name), pose_points.get(end_name), (255, 210, 0), 2)
    for start_name, end_name in HAND_DRAW_SEGMENTS:
        draw_segment(frame, left_hand.get(start_name), left_hand.get(end_name), (80, 220, 80), 2)
        draw_segment(frame, right_hand.get(start_name), right_hand.get(end_name), (0, 170, 255), 2)

    draw_points(frame, pose_points, (255, 210, 0), radius=4)
    draw_points(frame, left_hand, (80, 220, 80), radius=4)
    draw_points(frame, right_hand, (0, 170, 255), radius=4)


# Escolhe a janela ativa no timestamp atual, priorizando a mais severa.
def select_window_for_timestamp(timestamp_s: float, window_summaries: Iterable[Mapping[str, Any]]) -> dict[str, Any] | None:
    active_windows = [
        window
        for window in window_summaries
        if window.get('start_s', 0.0) <= timestamp_s < (window.get('end_s', 0.0) + 1e-9)
    ]
    if not active_windows:
        return None

    # Como as janelas se sobrepõem, transformamos várias candidatas em uma única janela de overlay:
    # primeiro priorizamos a mais severa, depois a mais recente.
    return max(
        active_windows,
        key=lambda window: (WINDOW_LEVEL_SEVERITY.get(window.get('window_level', 'insufficient_data'), -1), window.get('start_s', 0.0)),
    )


# Renderiza o painel textual com score, nivel da janela e explicacao curta.
def draw_status_panel(
    frame: np.ndarray,
    frame_record: Mapping[str, Any] | None,
    window_record: Mapping[str, Any] | None,
    *,
    exact_sample: bool,
) -> None:
    if frame_record is None:
        frame_record = {
            'frame_level': 'insufficient_data',
            'frame_score': 0.0,
            'active_rules': [],
            'explanation': 'insufficient data',
        }

    frame_level = frame_record.get('frame_level', 'insufficient_data')
    panel_color = FRAME_LEVEL_COLORS.get(frame_level, (120, 120, 120))
    active_rule_names = [pretty_rule_name(name) for name in frame_record.get('active_rules', [])]
    active_rule_names = [name for name in active_rule_names if name is not None]
    active_rules_text = ', '.join(active_rule_names) if active_rule_names else 'none'

    window_level = window_record.get('window_level', 'n/a') if window_record is not None else 'n/a'
    window_explanation = window_record.get('explanation', 'n/a') if window_record is not None else 'n/a'

    lines = [
        f"Frame level: {frame_level}",
        f"Frame score: {float(frame_record.get('frame_score', 0.0)):.2f}",
        f"Active rules: {active_rules_text}",
        f"Window level: {window_level}",
        f"Explanation: {frame_record.get('explanation', 'n/a')}",
    ]
    if frame_record.get('status') == 'insufficient_data':
        lines[0] = 'Frame level: insufficient data'
    if not exact_sample:
        lines.append('Overlay source: last sampled state')
    if window_record is not None:
        lines.append(f"Window note: {window_explanation}")

    overlay = frame.copy()
    panel_x, panel_y = 20, 20
    panel_width = min(780, frame.shape[1] - 40)
    panel_height = 34 + (30 * len(lines))
    cv2.rectangle(overlay, (panel_x, panel_y), (panel_x + panel_width, panel_y + panel_height), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0.0, frame)
    cv2.rectangle(frame, (panel_x, panel_y), (panel_x + panel_width, panel_y + panel_height), panel_color, 2)

    for line_idx, text in enumerate(lines, start=1):
        cv2.putText(
            frame,
            text,
            (panel_x + 12, panel_y + (line_idx * 26)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.68,
            (245, 245, 245),
            2,
            cv2.LINE_AA,
        )


# Faz a segunda passada no video original e grava o MP4 anotado final.
def render_annotated_video(
    source_path: Path,
    annotated_video_path: Path,
    frame_records: list[dict[str, Any]],
    drawings_by_frame: Mapping[int, dict[str, dict[str, PointData]]],
    window_summaries: list[dict[str, Any]],
    source_fps: float,
    frame_size: tuple[int, int],
) -> None:
    capture = cv2.VideoCapture(str(source_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not reopen source video for annotation: {source_path}')

    writer = cv2.VideoWriter(
        str(annotated_video_path),
        cv2.VideoWriter_fourcc(*'mp4v'),
        source_fps,
        frame_size,
    )
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f'Could not create annotated video: {annotated_video_path}')

    sorted_frames = sorted(frame_records, key=lambda frame: frame['frame_idx'])
    current_record: dict[str, Any] | None = None
    sample_cursor = 0
    frame_idx = 0

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break

            while sample_cursor < len(sorted_frames) and sorted_frames[sample_cursor]['frame_idx'] <= frame_idx:
                current_record = sorted_frames[sample_cursor]
                sample_cursor += 1

            exact_sample = current_record is not None and current_record['frame_idx'] == frame_idx
            if current_record is not None:
                drawing_points = drawings_by_frame.get(current_record['frame_idx'])
                if drawing_points is not None:
                    draw_landmarks_overlay(frame, drawing_points)
                window_record = select_window_for_timestamp(frame_idx / source_fps, window_summaries)
                draw_status_panel(frame, current_record, window_record, exact_sample=exact_sample)
            else:
                draw_status_panel(frame, None, None, exact_sample=False)

            writer.write(frame)
            frame_idx += 1
    finally:
        capture.release()
        writer.release()


## 11) Pipeline principal `process_video_asset(...)`

Esta é a célula de ponta a ponta. Ela abre o vídeo, calcula a amostragem, executa o MediaPipe Holistic, monta sinais por frame, agrega janelas, salva JSON e renderiza o MP4 anotado. O retorno final é o próprio payload serializável.


In [11]:
# Garante que o runtime tenha a dependencia principal antes do processamento.
def ensure_runtime_ready() -> None:
    if mp is None:
        raise ImportError('MediaPipe is not available. Execute the installation cell and rerun the imports.')
    if not hasattr(mp, 'tasks'):
        raise ImportError('The installed mediapipe package does not expose the Tasks API required by this notebook.')


# Resolve caminhos relativos de video a partir dos diretorios conhecidos do projeto.
def resolve_video_input_path(video_path: str | Path) -> Path:
    candidate = Path(video_path)
    if candidate.is_absolute():
        return candidate.resolve()

    search_roots = [Path.cwd(), REPO_ROOT, NOTEBOOK_ROOT]
    for root in search_roots:
        resolved = (root / candidate).resolve()
        if resolved.exists():
            return resolved
    return (NOTEBOOK_ROOT / candidate).resolve()


# Cria a task de pose em modo VIDEO usando o asset configurado no notebook.
def create_pose_landmarker() -> Any:
    base_options = mp.tasks.BaseOptions(model_asset_path=str(POSE_LANDMARKER_MODEL_PATH))
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    return mp.tasks.vision.PoseLandmarker.create_from_options(options)


# Cria a task de maos em modo VIDEO para detectar ate duas maos por frame.
def create_hand_landmarker() -> Any:
    base_options = mp.tasks.BaseOptions(model_asset_path=str(HAND_LANDMARKER_MODEL_PATH))
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


# Converte o frame RGB do OpenCV para o tipo de imagem esperado pelas tasks da MediaPipe.
def frame_to_mp_image(frame_rgb: np.ndarray) -> Any:
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


# Gera um payload seguro quando o video nao pode ser processado corretamente.
def build_fail_safe_payload(
    source_path: Path,
    json_output_path: Path,
    annotated_video_path: Path | None,
    *,
    error_message: str,
    source_fps: float = 0.0,
    duration_seconds: float = 0.0,
    frames_total: int = 0,
    frames_sampled: int = 0,
) -> dict[str, Any]:
    payload = {
        'schema_version': SCHEMA_VERSION,
        'video': {
            'video_id': source_path.stem or 'video',
            'source_path': str(source_path),
            'source_name': source_path.name,
            'annotated_video_path': str(annotated_video_path),
            'source_fps': optional_round(source_fps),
            'sample_fps': optional_round(TARGET_SAMPLE_FPS),
            'duration_seconds': optional_round(duration_seconds),
            'frames_total': int(frames_total),
            'frames_sampled': int(frames_sampled),
        },
        'pipeline': {
            'backend': PIPELINE_BACKEND,
            'coordinate_mode': COORDINATE_MODE_2D,
            'visibility_threshold': VISIBILITY_THRESHOLD,
            'ema_alpha': EMA_ALPHA,
            'window_seconds': WINDOW_SECONDS,
            'window_stride_seconds': WINDOW_STRIDE_SECONDS,
        },
        'limitations': [
            'This is a heuristic geometric posture screener based on MediaPipe pose and hand landmarks.',
            'The output reports observable posture and self-touch patterns only; it does not infer emotion, deception, diagnosis, or intent.',
            error_message,
        ],
        'frame_signals': [],
        'window_summaries': [],
        'video_summary': {
            'level': 'insufficient_data',
            'video_signal_ratio': None,
            'video_strong_window_ratio': None,
            'coverage_ratio': 0.0,
            'peak_window_level': 'insufficient_data',
            'most_common_trigger': None,
            'dominant_explanation': 'insufficient upper-body coverage for reliable screening',
            'explanation': 'insufficient upper-body coverage for reliable screening',
        },
    }
    write_payload_json(payload, json_output_path)
    return payload


# Executa o pipeline completo de analise, agregacao, anotacao e serializacao.
def process_video_asset(video_path: str | Path) -> dict[str, Any]:
    ensure_runtime_ready()
    assert_model_assets_available()

    source_path = resolve_video_input_path(video_path)

    video_id = source_path.stem or 'video'
    json_output_path = OUTPUT_JSON_DIR / f'{video_id}.signals.json'
    annotated_video_path = OUTPUT_VIDEO_DIR / f'{video_id}.annotated.mp4'

    if not source_path.exists():
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='Source video path does not exist.',
        )

    capture = cv2.VideoCapture(str(source_path))
    if not capture.isOpened():
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='OpenCV could not open the source video.',
        )

    source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0) or 30.0
    frames_total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration_seconds = (frames_total / source_fps) if source_fps > 0 else 0.0
    frame_step = max(1, round(source_fps / TARGET_SAMPLE_FPS))
    effective_sample_fps = source_fps / frame_step

    frame_records: list[dict[str, Any]] = []
    drawings_by_frame: dict[int, dict[str, dict[str, PointData]]] = {}
    coordinate_modes_used: Counter[str] = Counter()
    warnings: list[str] = []

    smoothers = build_smoothers()
    frame_idx = 0
    progress_bar = tqdm(total=frames_total or None, desc=f'Processing {video_id}', unit='frame')

    try:
        with create_pose_landmarker() as pose_landmarker, create_hand_landmarker() as hand_landmarker:
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                progress_bar.update(1)

                if frame_idx % frame_step != 0:
                    frame_idx += 1
                    continue

                timestamp_s = frame_idx / source_fps
                timestamp_ms = int(round(timestamp_s * 1000.0))
                try:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = frame_to_mp_image(frame_rgb)
                    pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
                    hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)
                    raw_sets = extract_landmark_sets(pose_result, hand_result)
                except Exception as exc:
                    warnings.append(f'Landmarker inference failed at sampled frame {frame_idx}: {exc}')
                    raw_sets = build_empty_landmark_sets()

                smoothed_sets = smooth_landmark_sets(frame_idx, raw_sets, smoothers)
                geometry = compute_reference_geometry(smoothed_sets)
                rules = evaluate_rules(geometry)
                frame_record = build_frame_signal_record(frame_idx, timestamp_s, geometry, rules)
                frame_records.append(frame_record)
                drawings_by_frame[frame_idx] = clone_drawing_points(smoothed_sets)
                coordinate_modes_used[geometry.get('coordinate_mode', COORDINATE_MODE_2D)] += 1

                frame_idx += 1
    finally:
        progress_bar.close()
        capture.release()

    if not frame_records:
        return build_fail_safe_payload(
            source_path,
            json_output_path,
            annotated_video_path,
            error_message='No sampled frames were produced from the source video.',
            source_fps=source_fps,
            duration_seconds=duration_seconds,
            frames_total=frames_total,
            frames_sampled=0,
        )

    window_summaries = aggregate_windows(frame_records, duration_seconds)
    video_summary = build_video_summary(frame_records, window_summaries)
    limitations = build_limitations(frame_records)
    limitations.extend(warnings)

    overall_coordinate_mode = coordinate_modes_used.most_common(1)[0][0] if coordinate_modes_used else COORDINATE_MODE_2D

    try:
        render_annotated_video(
            source_path=source_path,
            annotated_video_path=annotated_video_path,
            frame_records=frame_records,
            drawings_by_frame=drawings_by_frame,
            window_summaries=window_summaries,
            source_fps=source_fps,
            frame_size=(frame_width, frame_height),
        )
        annotated_video_string = str(annotated_video_path)
    except Exception as exc:
        limitations.append(f'Annotated video rendering failed: {exc}')
        annotated_video_string = None

    payload = {
        'schema_version': SCHEMA_VERSION,
        'video': {
            'video_id': video_id,
            'source_path': str(source_path),
            'source_name': source_path.name,
            'annotated_video_path': annotated_video_string,
            'source_fps': optional_round(source_fps),
            'sample_fps': optional_round(effective_sample_fps),
            'duration_seconds': optional_round(duration_seconds),
            'frames_total': int(frames_total),
            'frames_sampled': int(len(frame_records)),
        },
        'pipeline': {
            'backend': PIPELINE_BACKEND,
            'coordinate_mode': overall_coordinate_mode,
            'visibility_threshold': VISIBILITY_THRESHOLD,
            'ema_alpha': EMA_ALPHA,
            'window_seconds': WINDOW_SECONDS,
            'window_stride_seconds': WINDOW_STRIDE_SECONDS,
        },
        'limitations': limitations,
        'frame_signals': frame_records,
        'window_summaries': window_summaries,
        'video_summary': video_summary,
    }

    write_payload_json(payload, json_output_path)
    return payload


## 12) Execução em lote sobre `concepts_video/data/video`

A próxima célula varre a pasta de entrada, processa cada vídeo compatível e monta uma tabela resumo. 

In [12]:
VIDEO_PATHS = DEFAULT_VIDEO_PATHS

RUN_RESULTS: list[dict[str, Any]] = []
for index, video_path in enumerate(VIDEO_PATHS, start=1):
    display(Markdown(f'### Processing {index}/{len(VIDEO_PATHS)}: `{video_path.name}`'))
    payload = process_video_asset(video_path)
    RUN_RESULTS.append(payload)

SUMMARY_ROWS = [
    {
        'video_id': payload['video']['video_id'],
        'level': payload['video_summary']['level'],
        'coverage_ratio': payload['video_summary']['coverage_ratio'],
        'video_signal_ratio': payload['video_summary']['video_signal_ratio'],
        'strong_window_ratio': payload['video_summary']['video_strong_window_ratio'],
        'most_common_trigger': payload['video_summary']['most_common_trigger'],
        'json_path': str(OUTPUT_JSON_DIR / f"{payload['video']['video_id']}.signals.json"),
        'annotated_video_path': payload['video']['annotated_video_path'],
    }
    for payload in RUN_RESULTS
]

if pd is not None:
    SUMMARY_TABLE = pd.DataFrame(SUMMARY_ROWS)
else:
    SUMMARY_TABLE = SUMMARY_ROWS
display(SUMMARY_TABLE)


### Processing 1/4: `domestic_abuse1.mp4`

Processing domestic_abuse1:   0%|          | 0/384 [00:00<?, ?frame/s]

### Processing 2/4: `domestic_abuse2.mp4`

Processing domestic_abuse2:   0%|          | 0/431 [00:00<?, ?frame/s]

### Processing 3/4: `sad_woman1.mp4`

Processing sad_woman1:   0%|          | 0/877 [00:00<?, ?frame/s]

### Processing 4/4: `sad_woman2.mp4`

Processing sad_woman2:   0%|          | 0/917 [00:00<?, ?frame/s]

,video_id,level,coverage_ratio,video_signal_ratio,strong_window_ratio,most_common_trigger,json_path,annotated_video_path
0,domestic_abuse1,strong_signal,1.0,1.0000,0.8000,hand_on_face,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
1,domestic_abuse2,strong_signal,1.0,1.0000,1.0000,hand_on_face,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
2,sad_woman1,ok,1.0,0.0000,0.0000,NaN,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...
3,sad_woman2,strong_signal,1.0,0.9375,0.8125,hand_on_face,C:\Users\LuizAlbertodeAndrade\source\repos\scr...,C:\Users\LuizAlbertodeAndrade\source\repos\scr...


## 13) Inspeção rápida dos artefatos gerados

Esta etapa ajuda a validar o que foi produzido sem abrir o JSON manualmente. Ela mostra o resumo do primeiro resultado disponível e facilita localizar o MP4 anotado para revisão visual.


In [13]:
# Le um JSON de saida do notebook e devolve o dicionario correspondente.
def load_payload(json_path: str | Path) -> dict[str, Any]:
    return json.loads(Path(json_path).read_text(encoding='utf-8'))


if RUN_RESULTS:
    example_payload = RUN_RESULTS[0]
elif list(OUTPUT_JSON_DIR.glob('*.signals.json')):
    example_payload = load_payload(sorted(OUTPUT_JSON_DIR.glob('*.signals.json'))[0])
else:
    example_payload = None

if example_payload is None:
    display(Markdown('Run the batch cell above to generate JSON and annotated MP4 outputs.'))
else:
    display(Markdown('\n'.join([
        '### Example result',
        f"- Video: `{example_payload['video']['source_name']}`",
        f"- Video level: `{example_payload['video_summary']['level']}`",
        f"- Explanation: `{example_payload['video_summary']['explanation']}`",
        f"- JSON path: `{example_payload['video']['video_id']}.signals.json`",
        f"- Annotated video path: `{example_payload['video']['annotated_video_path']}`",
    ])))
    example_payload['video_summary']


### Example result
- Video: `domestic_abuse1.mp4`
- Video level: `strong_signal`
- Explanation: `hand-to-face contact`
- JSON path: `domestic_abuse1.signals.json`
- Annotated video path: `C:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\concepts_video\outputs\annotated_videos\domestic_abuse1.annotated.mp4`